In [2]:
import hail as hl
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

IndentationError: expected an indented block (analysis.py, line 126)

In [ ]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

In [ ]:
path1 = 'data/mt_new/ukb_wes_200k_annotated_chr21.mt'
mt1 = hl.read_matrix_table(path1)

In [ ]:
# load VEP annotation table
input_annotation_path = "data/vep/hail/ukb_wes_200k_chr1_vep.ht"
consequence_annotations = hl.read_table(input_annotation_path)
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 

In [29]:
mt1.filter_rows(mt1.)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'eur': int32
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        CM: array<float64>, 
        AN: int32
    }
    'consequence': struct {
        rsid: str, 
        qual: float64, 
        filters: set<str>, 
        info: struct {
            AF: float64, 
            AQ: array<int32>, 
            AC: int32, 
            AN: int32, 
            ExcessHet: float64
        }, 
        vep: struct {
            assembly_name: str, 
            allele_string: str, 
            ancestral: str, 
            colocated_variants: array<struct {
                aa_allele: str, 
                aa_maf: float64, 
                afr_allele: str, 
                afr_maf: float6

In [37]:
def gene_csqs_case_builder(in_mt):
    r''' Returns matrix table that contains gene consequence information from phased geneotypes.
    "": no alternate alleles,
    "HE": one alternate allele on either strand in a locus, 
    "HO": homozygous for alternate alleles
    "CH": two alternate allele on either strand in a locus (compound heterozygous)
    "CH+HO": two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    # create one gene_id for each item in gene_id array
    #in_mt = in_mt.explode_rows(in_mt.vep.worst_csq_by_gene_canonical)
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s0 = hl.agg.any(mt.a0_alt)))
    ht1 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s1 = hl.agg.any(mt.a1_alt)))
    ht2 = (in_mt.group_rows_by(in_mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var())))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.gene_id, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.gene_id, ht.s].hom_var)
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 'CH+HO')
           .when( (ht.s0) & (ht.s1), "CH")
           .when( (ht.hom_var), 'HO')
           .when( (ht.s0) & (ht.s1 == False), 'HE')
           .when( (ht.s1) & (ht.s0 == False), 'HE')
           .default(''))
    ht = ht.annotate_entries(csqs = expr)
    ht = ht.drop('s0').drop('s1').drop('hom_var')    
    return ht

In [63]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype

dtype('array<str>')

In [67]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype == hl.expr.ArrayExpression

False

In [71]:
hl.expr.ArrayExpression(hl.str(''))

TypeError: __init__() missing 1 required positional argument: 'type'

TypeError: __init__() missing 2 required positional arguments: 'x' and 'type'